## 🎯 Learning Objectives
* Understand the necessity and benefits of hybrid search in an e-commerce RAG system.
* Implement a sparse retrieval mechanism (e.g., BM25) for keyword matching.
* Implement a dense retrieval mechanism (e.g., vector similarity search using embeddings and FAISS) for semantic understanding.
* Combine sparse and dense retrieval results using a re-ranking strategy like Reciprocal Rank Fusion (RRF).
* Evaluate the effectiveness of a hybrid retrieval layer for e-commerce product search queries.


## Exercise: Building the Retrieval Layer with Hybrid Search

In an e-commerce context, users often search using a mix of exact keywords (e.g., "iPhone 15 Pro Max 256GB") and more semantic, descriptive queries (e.g., "a durable phone for outdoor activities"). A purely keyword-based search might miss semantically relevant products, while a purely semantic search might struggle with highly specific product codes or names. This is where **hybrid search** becomes crucial.

Hybrid search combines the strengths of both sparse retrieval (like BM25, which excels at keyword matching) and dense retrieval (like vector similarity search, which captures semantic meaning). By integrating these two approaches, we can achieve a more robust and accurate retrieval layer for our Agentic RAG system.

### Task

Your task is to implement a `hybrid_search` function that effectively combines sparse and dense retrieval methods to find relevant products from a mock e-commerce catalog. You will use a pre-defined dataset and set up the necessary components for both retrieval types.

### Requirements

1.  **Sparse Retrieval**: Implement a BM25-like search to find products based on keyword overlap. You can use the `rank_bm25` library.
2.  **Dense Retrieval**: Implement a vector similarity search using a pre-trained sentence transformer model (`BAAI/bge-small-en-v1.5`) and a FAISS index to find semantically similar products.
3.  **Result Combination**: Combine the results from both sparse and dense retrievers using **Reciprocal Rank Fusion (RRF)**. This method is effective for merging ranked lists without requiring relevance scores to be on the same scale.
4.  **Function Signature**: Your primary function should be named `hybrid_search` and accept a `query` string and an optional `top_k` integer (defaulting to 5) for the number of final results.
5.  **Output**: The `hybrid_search` function should return a list of dictionaries, where each dictionary represents a product and includes its `product_id`, `name`, and `description`.

### Evaluation Criteria

*   **Correctness**: The `hybrid_search` function correctly implements both sparse and dense retrieval components.
*   **Integration**: Results from both methods are combined using RRF.
*   **Clarity**: The code is well-structured, readable, and includes comments where necessary.
*   **Effectiveness**: The function returns relevant products for diverse queries (keyword-heavy and semantic).
*   **Efficiency**: The solution demonstrates a reasonable approach to performance for the given dataset size.


In [ ]:
# Install necessary libraries (if not already installed)
# !pip install pandas numpy sentence-transformers rank_bm25 faiss-cpu

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
import random

# --- Mock E-commerce Product Dataset ---
products_data = [
    {"product_id": "P001", "name": "Smartphone X Pro", "description": "Flagship smartphone with 120Hz OLED display, 108MP camera, and A17 Bionic chip. 256GB storage. Ideal for photography and gaming.", "category": "Electronics"},
    {"product_id": "P002", "name": "Ultra HD 4K Smart TV 65 inch", "description": "Immersive 65-inch 4K UHD Smart TV with HDR10+, Dolby Vision, and voice control. Perfect for home cinema.", "category": "Electronics"},
    {"product_id": "P003", "name": "Wireless Noise-Cancelling Headphones", "description": "Premium over-ear headphones with active noise cancellation, 30-hour battery life, and crystal-clear audio. Great for travel and focus.", "category": "Audio"},
    {"product_id": "P004", "name": "Ergonomic Office Chair", "description": "Adjustable ergonomic chair with lumbar support, breathable mesh, and 3D armrests. Designed for long hours of comfortable work.", "category": "Office Furniture"},
    {"product_id": "P005", "name": "Smartwatch Z Series", "description": "Fitness tracker and smartwatch with heart rate monitor, GPS, NFC for payments, and waterproof design. Compatible with iOS and Android.", "category": "Wearables"},
    {"product_id": "P006", "name": "Portable Bluetooth Speaker", "description": "Compact and powerful Bluetooth speaker with 15-hour battery, IPX7 waterproof rating, and rich bass. Perfect for outdoor parties.", "category": "Audio"},
    {"product_id": "P007", "name": "Gaming Laptop G15", "description": "High-performance gaming laptop with RTX 4070 GPU, Intel i9 processor, 32GB RAM, and 1TB SSD. 144Hz display. For serious gamers.", "category": "Computers"},
    {"product_id": "P008", "name": "Espresso Machine Deluxe", "description": "Automatic espresso machine with integrated grinder, milk frother, and customizable settings. Enjoy barista-quality coffee at home.", "category": "Home Appliances"},
    {"product_id": "P009", "name": "Robot Vacuum Cleaner", "description": "Smart robot vacuum with LiDAR navigation, app control, and automatic charging. Cleans carpets and hard floors efficiently.", "category": "Home Appliances"},
    {"product_id": "P010", "name": "External SSD 2TB USB-C", "description": "Ultra-fast 2TB external solid-state drive with USB-C 3.2 Gen 2. Ideal for large file transfers and backups. Compact and durable.", "category": "Storage"},
    {"product_id": "P011", "name": "Noise-Cancelling Earbuds", "description": "True wireless earbuds with active noise cancellation, transparency mode, and secure fit. Great for commuting and workouts.", "category": "Audio"},
    {"product_id": "P012", "name": "Curved Gaming Monitor 27 inch", "description": "27-inch 1440p curved gaming monitor with 165Hz refresh rate and 1ms response time. AMD FreeSync Premium compatible.", "category": "Computers"},
    {"product_id": "P013", "name": "Smart Home Security Camera", "description": "Wireless indoor security camera with 1080p video, night vision, two-way audio, and motion detection. Cloud storage optional.", "category": "Smart Home"},
    {"product_id": "P014", "name": "Electric Standing Desk", "description": "Motorized standing desk with memory presets, spacious desktop, and sturdy steel frame. Improve posture and productivity.", "category": "Office Furniture"},
    {"product_id": "P015", "name": "Portable Power Bank 20000mAh", "description": "High-capacity power bank with 20000mAh, USB-C PD, and Quick Charge 3.0. Charges multiple devices quickly. Essential for travel.", "category": "Accessories"}
]

products_df = pd.DataFrame(products_data)

# Combine name and description for better search context
products_df['search_text'] = products_df['name'] + ". " + products_df['description']

# --- Initialize Dense Retriever (Sentence Transformer) ---
print("Loading Sentence Transformer model...")
# Using a modern, performant embedding model
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print("Model loaded.")

# Generate embeddings for all product search texts
print("Generating product embeddings...")
corpus_embeddings = embedding_model.encode(products_df['search_text'].tolist(), convert_to_tensor=True)
corpus_embeddings_np = corpus_embeddings.cpu().numpy()
print(f"Generated {len(corpus_embeddings_np)} embeddings.")

# --- Initialize FAISS Index for Dense Retrieval ---
embedding_dim = corpus_embeddings_np.shape[1]
faiss_index = faiss.IndexFlatL2(embedding_dim) # L2 distance for similarity
faiss_index.add(corpus_embeddings_np)
print(f"FAISS index created with {faiss_index.ntotal} vectors.")

# --- Initialize Sparse Retriever (BM25) ---
# Tokenize the corpus for BM25
# A simple tokenizer for demonstration. For production, consider more advanced NLP tokenizers.
tokenized_corpus = [doc.lower().split() for doc in products_df['search_text'].tolist()]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 retriever initialized.")

print("\nSetup complete. You can now proceed with implementing the hybrid search function.")


### Your Implementation: Hybrid Search Function

Now it's your turn to implement the `hybrid_search` function. Use the pre-initialized `embedding_model`, `faiss_index`, `bm25` object, and `products_df` to perform the hybrid search.

Remember the steps:
1.  Perform sparse retrieval using `bm25.get_scores()`.
2.  Perform dense retrieval by encoding the query and searching the `faiss_index`.
3.  Combine the ranked lists using Reciprocal Rank Fusion (RRF).
4.  Return the top `k` unique products.

```python
def hybrid_search(query: str, top_k: int = 5) -> list[dict]:
    # Your code here
    pass

# Example Usage (after your implementation):
# query_1 = "best phone for photos"
# results_1 = hybrid_search(query_1, top_k=3)
# print(f"\nQuery: '{query_1}'")
# for r in results_1:
#     print(f"- {r['name']} (ID: {r['product_id']})")

# query_2 = "gaming laptop with rtx 4070"
# results_2 = hybrid_search(query_2, top_k=3)
# print(f"\nQuery: '{query_2}'")
# for r in results_2:
#     print(f"- {r['name']} (ID: {r['product_id']})")
```


In [ ]:
### Reference Solution: Hybrid Search Function

This solution demonstrates how to implement the `hybrid_search` function using the pre-initialized components. It includes sparse retrieval with BM25, dense retrieval with Sentence Transformers and FAISS, and combines results using Reciprocal Rank Fusion (RRF).


In [ ]:
def reciprocal_rank_fusion(ranked_lists: list[list[int]], k: int = 60) -> list[int]:
    """
    Performs Reciprocal Rank Fusion (RRF) on multiple ranked lists.
    Args:
        ranked_lists: A list of lists, where each inner list contains document indices
                      ranked by a specific retriever.
        k: A constant used in the RRF formula (1 / (k + rank)). Higher k means
           lower ranks contribute more.
    Returns:
        A list of document indices, globally ranked by RRF score.
    """
    fused_scores = {}
    for ranked_list in ranked_lists:
        for rank, doc_idx in enumerate(ranked_list):
            # RRF formula: 1 / (k + rank)
            score = 1.0 / (k + rank + 1)  # +1 because rank is 0-indexed
            fused_scores[doc_idx] = fused_scores.get(doc_idx, 0.0) + score

    # Sort documents by their fused scores in descending order
    sorted_docs = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)
    return [doc_idx for doc_idx, _ in sorted_docs]

def hybrid_search(query: str, top_k: int = 5) -> list[dict]:
    """
    Performs a hybrid search combining BM25 (sparse) and vector similarity (dense)
    retrieval, then fuses the results using Reciprocal Rank Fusion (RRF).

    Args:
        query: The search query string.
        top_k: The number of top unique products to return.

    Returns:
        A list of dictionaries, each representing a product with 'product_id', 'name',
        and 'description'.
    """
    # --- 1. Sparse Retrieval (BM25) ---
    # Tokenize the query for BM25
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # Get BM25 top N results (e.g., top 100 for RRF input)
    # We need indices, not just scores. Sort by score and get original indices.
    bm25_ranked_indices = np.argsort(bm25_scores)[::-1].tolist()
    # Limit to a reasonable number for RRF, e.g., 100 or all if corpus is small
    bm25_ranked_indices = bm25_ranked_indices[:min(len(bm25_ranked_indices), 100)]

    # --- 2. Dense Retrieval (Vector Similarity Search) ---
    # Encode the query into a vector
    query_embedding = embedding_model.encode(query, convert_to_tensor=True).cpu().numpy()
    query_embedding = np.expand_dims(query_embedding, axis=0) # FAISS expects 2D array

    # Search the FAISS index
    # D: distances, I: indices
    D, I = faiss_index.search(query_embedding, k=min(faiss_index.ntotal, 100)) # Get top 100 for RRF
    dense_ranked_indices = I[0].tolist()

    # --- 3. Combine Results using Reciprocal Rank Fusion (RRF) ---
    # Ensure both lists are of document indices (integers corresponding to products_df.index)
    fused_ranked_indices = reciprocal_rank_fusion([bm25_ranked_indices, dense_ranked_indices])

    # --- 4. Retrieve Top K Unique Products ---
    final_results = []
    seen_product_ids = set()

    for doc_idx in fused_ranked_indices:
        product = products_df.iloc[doc_idx]
        if product['product_id'] not in seen_product_ids:
            final_results.append({
                "product_id": product['product_id'],
                "name": product['name'],
                "description": product['description']
            })
            seen_product_ids.add(product['product_id'])
            if len(final_results) >= top_k:
                break

    return final_results

# --- Example Usage ---
print("\n--- Testing Hybrid Search ---")

# Query 1: Keyword-heavy, specific product
query_1 = "Smartphone X Pro 256GB"
print(f"\nQuery: '{query_1}'")
results_1 = hybrid_search(query_1, top_k=3)
for i, r in enumerate(results_1):
    print(f"{i+1}. {r['name']} (ID: {r['product_id']})")

# Query 2: Semantic, descriptive
query_2 = "durable headphones for travel with noise cancellation"
print(f"\nQuery: '{query_2}'")
results_2 = hybrid_search(query_2, top_k=3)
for i, r in enumerate(results_2):
    print(f"{i+1}. {r['name']} (ID: {r['product_id']})")

# Query 3: Mixed, looking for a specific feature in a category
query_3 = "gaming monitor with high refresh rate"
print(f"\nQuery: '{query_3}'")
results_3 = hybrid_search(query_3, top_k=3)
for i, r in enumerate(results_3):
    print(f"{i+1}. {r['name']} (ID: {r['product_id']})")

# Query 4: General category search
query_4 = "smart home devices"
print(f"\nQuery: '{query_4}'")
results_4 = hybrid_search(query_4, top_k=3)
for i, r in enumerate(results_4):
    print(f"{i+1}. {r['name']} (ID: {r['product_id']})")

# Query 5: Product ID search (should be caught by BM25)
query_5 = "P007"
print(f"\nQuery: '{query_5}'")
results_5 = hybrid_search(query_5, top_k=1)
for i, r in enumerate(results_5):
    print(f"{i+1}. {r['name']} (ID: {r['product_id']})")
